# HW1: Integration

**NTUT MuJoCo Course - Lecture 2**

Everything you do later in this course - simulating a robot, solving a trajectory
optimization problem, running a controller in MuJoCo - sits on top of a numerical
integrator. In this homework you implement six of them and see, concretely, how that
choice changes what your simulation predicts.

| Part | Points | Topic |
| --- | --- | --- |
| A | 10 | Explicit integration: forward Euler, midpoint, RK4 |
| B | 10 | Implicit integration: three residuals + Newton's method |
| C | 5 | Accuracy comparison and short answers |
| D | +5 bonus | Reusing your integrators on a cart-pole |

### Rules

- **DO NOT USE `while` LOOPS** anywhere in this homework.
- **DO NOT ALTER THE TEST CELLS.** They are marked `TEST CELL`, and we check.
- Do not change `dt`, `tf`, or `x0` inside a test cell. The double pendulum is chaotic and
  the tolerances are tied to those exact values.
- Every function you must write raises `NotImplementedError` until you replace the `# TODO`.
  Search this notebook for `TODO` to find them all.
- Before submitting: **Kernel -> Restart Kernel and Run All Cells**. A notebook that only
  passes when run out of order does not pass for us.

In [ ]:
%matplotlib inline
import numpy as np
import casadi as ca
import matplotlib.pyplot as plt
from dataclasses import dataclass
from IPython.display import display

from animation import animate_pendulum, animate_cartpole, show_animation

np.set_printoptions(precision=6, suppress=True)
print('imports ok')

## Python and CasADi warmup

Two things about Python and CasADi cause almost every bug students hit in this homework.
Both cells below are given and already work - read them before you start.

In [ ]:
# 1. Assignment does NOT copy an array.
x = np.array([1.0, 2.0, 3.0])
y = x                 # NEVER DO THIS - y and x are now the SAME array
y[2] = 100.0
print('x =', x, '  <- editing y also edited x')

y = 1 * x             # arithmetic builds a new array
y = x.copy()          # or say so explicitly
y[0] = -1.0
print('x =', x, '  <- x is untouched now')

# 2. The one that will actually bite you: a ROW of a 2-D array is a VIEW, not a copy.
A = np.zeros((3, 4))
row = A[0]            # points into A
row += 5.0            # in-place, so this writes into A
print('A[0] =', A[0], ' <- writing through the view edited A')

In [ ]:
# 3. CasADi functions hand back CasADi matrices, and they are COLUMN vectors.
v = ca.vertcat(1.0, 2.0, 3.0, 4.0)      # this is what dynamics(...) returns
print(type(v).__name__, 'with shape', v.shape, '  <- (4, 1), not (4,)')

X = np.zeros((2, 4))
try:
    X[1] = v
except ValueError as error:
    print('\nassigning it straight into a numpy row fails:\n   ', error)


def vector(value):
    """CasADi matrix (or anything array-like) -> flat 1-D numpy array."""
    return np.asarray(value, dtype=float).reshape(-1)


X[1] = vector(v)
print('\nwith vector():  X[1] =', X[1])

# Note: your integrators are called with plain numpy arrays, so you may use np.sin,
# float(), if-statements and anything else you like inside them.

## Q1: Integration (25 pts)

We are given continuous-time dynamics

$$\dot{x} = f(x)$$

which in code is `xdot = dynamics(params, x)`. An *integrator* answers the question this
leaves open: given the state $x_k$ at time $t_k$, what is the state $x_{k+1}$ one time step
$\Delta t$ later?

Every function you write below takes `params` and `dynamics` as **arguments** instead of
reaching for a global. That is not busywork - in Part D you will run these exact same six
integrators on a completely different robot just by passing different arguments.

Our test system is a double pendulum, with state

$$x = \begin{bmatrix} \theta_1 & \dot\theta_1 & \theta_2 & \dot\theta_2 \end{bmatrix}^T$$

It is chaotic, and it conserves energy. That makes it an unusually honest test: it has no
friction and no motor, so **any** energy your simulation gains or loses is numerical error,
and you can see it on a plot.

### Part A (10 pts): Explicit Integration

An **explicit** integrator computes $x_{k+1}$ directly from $x_k$: evaluate the right-hand
side, and you are done. We will implement three.

- Forward Euler (1st order)
- Midpoint, a.k.a. RK2 (2nd order)
- RK4 (4th order)

In [ ]:
# these functions are given, no TODOs here

@dataclass(frozen=True)
class Parameters:
    m1: float = 1.0
    m2: float = 1.0
    L1: float = 1.0
    L2: float = 1.0
    g: float = 9.8


PARAMS = Parameters()
X0 = np.array([np.pi / 1.6, 0.0, np.pi / 1.8, 0.0])


def double_pendulum_dynamics(params, x):
    """Continuous equations of motion. State is [theta1, omega1, theta2, omega2]."""
    theta1, omega1, theta2, omega2 = (x[i] for i in range(4))
    m1, m2, L1, L2, g = params.m1, params.m2, params.L1, params.L2, params.g
    c, s = ca.cos(theta1 - theta2), ca.sin(theta1 - theta2)
    alpha1 = (m2 * g * ca.sin(theta2) * c
              - m2 * s * (L1 * c * omega1**2 + L2 * omega2**2)
              - (m1 + m2) * g * ca.sin(theta1)) / (L1 * (m1 + m2 * s**2))
    alpha2 = ((m1 + m2) * (L1 * omega1**2 * s - g * ca.sin(theta2)
                           + g * ca.sin(theta1) * c)
              + m2 * L2 * omega2**2 * s * c) / (L2 * (m1 + m2 * s**2))
    return ca.vertcat(omega1, alpha1, omega2, alpha2)


def double_pendulum_energy(params, x):
    """Kinetic + potential energy, retaining 2 m pivot height."""
    theta1, omega1, theta2, omega2 = (x[i] for i in range(4))
    m1, m2, L1, L2, g = params.m1, params.m2, params.L1, params.L2, params.g
    kinetic = (0.5 * (m1 + m2) * L1**2 * omega1**2
               + 0.5 * m2 * L2**2 * omega2**2
               + m2 * L1 * L2 * omega1 * omega2 * ca.cos(theta1 - theta2))
    potential = ((m1 + m2) * g * (2 - L1 * ca.cos(theta1))
                 - m2 * g * L2 * ca.cos(theta2))
    return kinetic + potential


def time_grid(dt, tf):
    """Time samples 0, dt, 2*dt, ... up to tf. Use this rather than rolling your own."""
    if not np.isfinite(dt) or not np.isfinite(tf) or dt <= 0 or tf < 0:
        raise ValueError('dt must be positive and tf nonnegative, both finite')
    # Like Julia's 0:dt:tf, omit any fractional final step.
    return np.arange(int(np.floor(tf / dt + 1e-10)) + 1) * dt


def max_err_E(E):
    """Largest deviation of an energy history from its initial value."""
    return float(np.max(np.abs(E - E[0])))


print('energy at x0:', float(double_pendulum_energy(PARAMS, X0)), 'J')
print('time_grid(0.01, 2.0) has', len(time_grid(0.01, 2.0)), 'points')

#### Forward Euler

The simplest possible choice: pretend $\dot{x}$ is constant across the step.

$$x_{k+1} = x_k + \Delta t \, f(x_k) \quad \quad \text{(Forward Euler)}$$

In [ ]:
def forward_euler(params, dynamics, x, dt):
    """x_next = forward_euler(params, dynamics, x, dt)

    Given `xdot = dynamics(params, x)`, take the current state `x` and integrate it
    forward by `dt` using the Forward Euler method.
    """
    # xdot = dynamics(params, x)
    # TODO: implement forward euler
    raise NotImplementedError('forward euler not implemented')

Now simulate with it. Write the loop by hand this one time - we will generalise it into a
function in a moment.

In [ ]:
dt, tf = 0.01, 30.0
times = time_grid(dt, tf)
X = np.zeros((len(times), len(X0)))
X[0] = 1 * X0          # a copy, so we can never write through into X0

# TODO: simulate the double pendulum forward with `forward_euler`
# X[k] is x_k, so X[k + 1] = vector(forward_euler(PARAMS, double_pendulum_dynamics, X[k], dt))
# DO NOT USE A WHILE LOOP


# ------------------------------------------------------------------- TEST CELL (A1)
E = np.array([float(double_pendulum_energy(PARAMS, x)) for x in X])
assert X.shape == (3001, 4), f'expected (3001, 4) states, got {X.shape}'
assert np.allclose(X[0], X0), 'X[0] is no longer the initial condition'
assert np.linalg.norm(X[-1]) > 1e-10, 'X was never updated - did your loop run?'
assert 2 < E[-1] / E[0] < 3, f'energy ratio {E[-1] / E[0]:.4f} is outside the expected range'
print(f'Part A1 tests passed. Energy went from {E[0]:.3f} J to {E[-1]:.3f} J.')

fig, axes = plt.subplots(2, 1, sharex=True, figsize=(9, 7))
axes[0].plot(times, X)
axes[0].legend(['theta1', 'omega1', 'theta2', 'omega2'])
axes[0].set(ylabel='State', title='Double pendulum: Forward Euler, dt = 0.01 s')
axes[1].plot(times, E)
axes[1].set(xlabel='Time (s)', ylabel='Energy (J)')
fig.tight_layout()
plt.show()

This pendulum has no friction and no motor, so its total energy should be **constant**.
Instead forward Euler more than doubles it in 30 seconds, and the state plot shows the
pendulum spinning faster and faster.

That is not a bug in your code. Forward Euler steps along the tangent, and on a curved
trajectory the tangent always lands slightly outside the true arc - which here means
slightly more energy, every single step. Watch it happen:

In [ ]:
# a 10 second window, so the embedded animation stays small
display(show_animation(animate_pendulum(PARAMS, X[:1001], dt,
                                        title='Forward Euler: double pendulum')))

#### Midpoint and RK4

We can do far better by sampling $f$ more than once per step. The midpoint method takes a
half step, evaluates the dynamics *there*, and uses that slope for the whole step:

$$
\begin{align}
x_m &= x_k + \tfrac{1}{2} \Delta t \, f(x_k) \\
x_{k+1} &= x_k + \Delta t \, f(x_m) \quad \quad \text{(Midpoint, 2nd order)}
\end{align}
$$

RK4 pushes the same idea to four evaluations and a weighted average:

$$
\begin{align}
k_1 &= \Delta t \, f(x_k) \\
k_2 &= \Delta t \, f(x_k + k_1 / 2) \\
k_3 &= \Delta t \, f(x_k + k_2 / 2) \\
k_4 &= \Delta t \, f(x_k + k_3) \\
x_{k+1} &= x_k + \tfrac{1}{6} \left( k_1 + 2 k_2 + 2 k_3 + k_4 \right) \quad \quad \text{(RK4, 4th order)}
\end{align}
$$

In [ ]:
def midpoint(params, dynamics, x, dt):
    """x_next = midpoint(params, dynamics, x, dt)"""
    # TODO: implement explicit midpoint
    raise NotImplementedError('midpoint not implemented')


def rk4(params, dynamics, x, dt):
    """x_next = rk4(params, dynamics, x, dt)"""
    # TODO: implement RK4
    raise NotImplementedError('rk4 not implemented')

Next, generalise the loop you wrote above so that *any* explicit integrator can be dropped
into it. Note the `dynamics` and `energy_fn` arguments: use them, do not hardcode the double
pendulum, or Part D will not work.

In [ ]:
def simulate_explicit(params, dynamics, integrator, x0, dt, tf,
                      energy_fn=double_pendulum_energy):
    """X, E = simulate_explicit(params, dynamics, integrator, x0, dt, tf)

    Roll the dynamics forward from `x0` to time `tf` with an explicit `integrator`.
    Returns the state history X (one state per row) and the energy history E.
    """
    times = time_grid(dt, tf)
    X = np.zeros((len(times), len(x0)))
    X[0] = 1 * np.asarray(x0, dtype=float)      # given: a copy, do not alias x0

    # TODO: simulate X forward with the explicit integrator
    # X[k] is x_k, so X[k + 1] = vector(integrator(params, dynamics, X[k], dt))
    # DO NOT USE A WHILE LOOP

    E = np.array([float(energy_fn(params, x)) for x in X])      # given
    return X, E

In [ ]:
# ------------------------------------------------------------------- TEST CELL (A2)
# do not change dt, tf or the initial condition in this cell
dt, tf = 0.01, 2.0
explicit = (forward_euler, midpoint, rk4)
results = {}
for method in explicit:
    X, E = simulate_explicit(PARAMS, double_pendulum_dynamics, method, X0, dt, tf)
    results[method] = (X, E)
    assert X.shape == (201, 4), f'{method.__name__}: expected (201, 4), got {X.shape}'
    assert np.all(np.isfinite(X)), f'{method.__name__}: produced non-finite states'
    assert np.allclose(X[0], X0), f'{method.__name__}: X[0] is no longer x0'

dE = {method: E[-1] - E[0] for method, (_, E) in results.items()}
assert abs(dE[midpoint]) < 1e-1, f'midpoint energy drift {dE[midpoint]:.3e} is too large'
assert abs(dE[rk4]) < 1e-4, f'rk4 energy drift {dE[rk4]:.3e} is too large'

# Observed order of accuracy: halve dt, see how much the error shrinks.
reference = simulate_explicit(PARAMS, double_pendulum_dynamics, rk4, X0, 2.5e-4, 1.0)[0][-1]
order = {}
print(f'{"integrator":<16}{"err(2e-2)":>12}{"err(1e-2)":>12}{"err(5e-3)":>12}{"order":>8}')
for method in explicit:
    errors = [np.linalg.norm(simulate_explicit(
        PARAMS, double_pendulum_dynamics, method, X0, h, 1.0)[0][-1] - reference)
        for h in (2e-2, 1e-2, 5e-3)]
    order[method] = float(np.log2(errors[-2] / errors[-1]))
    print(f'{method.__name__:<16}' + ''.join(f'{e:12.3e}' for e in errors)
          + f'{order[method]:8.2f}')

assert 0.8 < order[forward_euler] < 1.9, 'forward euler should look roughly 1st order'
assert 1.8 < order[midpoint] < 2.2, f'midpoint looks {order[midpoint]:.2f} order, expected 2'
assert 3.7 < order[rk4] < 4.3, f'rk4 looks {order[rk4]:.2f} order, expected 4'
print('\nPart A2 tests passed.')

### Part B (10 pts): Implicit Integrators

Every method in Part A produced $x_{k+1}$ directly. An **implicit** method instead gives
you an equation that $x_{k+1}$ has to satisfy, with $x_{k+1}$ appearing on *both* sides:

$$x_{k+1} = x_k + \Delta t \, f(x_{k+1}) \quad \quad \text{(Backward Euler)}$$

We cannot simply evaluate that. Instead we move everything to one side and define a
**residual** $r$, which is zero exactly when $x_{k+1}$ is correct. Writing $x_1$ for $x_k$
and $x_2$ for $x_{k+1}$:

$$
\begin{align}
r_{BE}(x_1, x_2, \Delta t) &= x_1 + \Delta t \, f(x_2) - x_2 \quad \quad \text{(Backward Euler)} \\
r_{IM}(x_1, x_2, \Delta t) &= x_1 + \Delta t \, f\!\left(\tfrac{x_1 + x_2}{2}\right) - x_2 \quad \quad \text{(Implicit Midpoint)}
\end{align}
$$

Hermite-Simpson is the same idea with a cubic interpolation across the step. It builds a
midpoint from both endpoints and both slopes, then applies Simpson's rule:

$$
\begin{align}
f_1 &= f(x_1), \quad f_2 = f(x_2) \\
x_m &= \frac{x_1 + x_2}{2} + \frac{\Delta t}{8} \left( f_1 - f_2 \right) \\
r_{HS}(x_1, x_2, \Delta t) &= x_1 + \frac{\Delta t}{6} \left( f_1 + 4 f(x_m) + f_2 \right) - x_2 \quad \quad \text{(Hermite-Simpson)}
\end{align}
$$

In the next cell you write **only these three residual functions**.

In [ ]:
# NOTE: we are NOT solving anything here, simply return the residual.
# Each function takes BOTH x1 and x2 and returns a vector that is zero when x2 is correct.

def backward_euler(params, dynamics, x1, x2, dt):
    """residual = backward_euler(params, dynamics, x1, x2, dt)"""
    # TODO: implement the backward euler residual
    raise NotImplementedError('backward euler not implemented')


def implicit_midpoint(params, dynamics, x1, x2, dt):
    """residual = implicit_midpoint(params, dynamics, x1, x2, dt)"""
    # TODO: implement the implicit midpoint residual
    raise NotImplementedError('implicit midpoint not implemented')


def hermite_simpson(params, dynamics, x1, x2, dt):
    """residual = hermite_simpson(params, dynamics, x1, x2, dt)"""
    # TODO: implement the hermite simpson residual
    raise NotImplementedError('hermite simpson not implemented')

In [ ]:
# ------------------------------------------------------------------- TEST CELL (B1)
# These check your residuals directly, so you can debug them before writing Newton.
implicit = (backward_euler, implicit_midpoint, hermite_simpson)
x1 = np.array([0.1, 0.2, 0.3, 0.4])

# 1. With dt = 0 and x2 = x1, "stay put" is the exact answer, so every residual vanishes.
for method in implicit:
    r = np.asarray(method(PARAMS, double_pendulum_dynamics, ca.DM(x1), ca.DM(x1), 0.0)).ravel()
    assert np.linalg.norm(r) < 1e-12, \
        f'{method.__name__}: residual should be zero when dt = 0 and x2 = x1'

# 2. Known values at a fixed test point. A flipped sign (x2 - x1 - dt*f) fails here.
x2 = np.array([0.15, 0.25, 0.35, 0.45])
expected = {
    backward_euler:    [-0.0250000000, -0.0099000513, -0.0050000000, -0.4265821541],
    implicit_midpoint: [-0.0275000000,  0.0143565723, -0.0075000000, -0.4270020992],
    hermite_simpson:   [-0.0270956410,  0.0148065118, -0.0075056566, -0.4270642537],
}
for method in implicit:
    r = np.asarray(method(PARAMS, double_pendulum_dynamics, ca.DM(x1), ca.DM(x2), 0.1)).ravel()
    assert r.shape == (4,), f'{method.__name__}: expected 4 residuals, got shape {r.shape}'
    np.testing.assert_allclose(r, expected[method], atol=1e-9,
                               err_msg=f'{method.__name__} residual is wrong')
    print(f'{method.__name__:<20} {r}')
print('\nPart B1 tests passed.')

#### Driving the residual to zero with Newton's method

A residual is only useful if we can solve it. Given $x_1$ and $\Delta t$, we want the $x_2$
with $r(x_2) = 0$. Newton's method linearises $r$ about the current guess and jumps to where
that linearisation crosses zero:

$$x_2 \leftarrow x_2 - \left( \frac{\partial r}{\partial x_2} \right)^{-1} r(x_2)$$

We start from $x_2 = x_1$ and repeat until $\|r\|$ drops below a tolerance. Because the
initial guess is only one time step away from the answer, this converges in a handful of
iterations - typically three or four here.

That leaves the Jacobian $\partial r / \partial x_2$. We will **not** finite-difference it.
CasADi differentiates your residual symbolically and produces the exact Jacobian. The given
function below compiles one `ca.Function` returning **both** the residual and its Jacobian
in a single call, which is why we build it once and reuse it for every step.

In [ ]:
# this function is given, no TODOs here

def build_implicit_functions(params, dynamics, integrator, size):
    """Compile an integrator's residual and its exact Jacobian into one callable.

    Returns a `ca.Function` used as:  residual, jacobian = functions(x1, x2, dt)
    """
    x1, x2, dt = ca.SX.sym('x1', size), ca.SX.sym('x2', size), ca.SX.sym('dt')
    residual = integrator(params, dynamics, x1, x2, dt)
    return ca.Function(integrator.__name__ + '_residual', [x1, x2, dt],
                       [residual, ca.jacobian(residual, x2)])


demo = build_implicit_functions(PARAMS, double_pendulum_dynamics, backward_euler, 4)
residual, jacobian = demo(np.array([0.1, 0.2, 0.3, 0.4]), np.array([0.15, 0.25, 0.35, 0.45]), 0.1)
print('residual', residual.shape, ' jacobian', jacobian.shape, '\n')
print(jacobian)

In [ ]:
def implicit_integrator_solve(params, dynamics, integrator, x1, dt,
                              tol=1e-13, max_iters=10, functions=None):
    """x2 = implicit_integrator_solve(params, dynamics, integrator, x1, dt)

    Use Newton's method to find the x2 that drives `integrator`'s residual to zero.
    """
    if functions is None:                              # given
        functions = build_implicit_functions(params, dynamics, integrator, len(x1))

    x2 = 1 * np.asarray(x1, dtype=float)               # given: a COPY of x1, not an alias

    # DO NOT USE A WHILE LOOP
    for i in range(max_iters):
        residual, jacobian = functions(x1, x2, dt)     # given
        residual = np.asarray(residual).ravel()        # given: (4, 1) -> (4,)

        # TODO: return x2 as soon as np.linalg.norm(residual) is below tol

        # TODO: otherwise take one Newton step.
        # hint: np.linalg.solve(np.asarray(jacobian), residual) is the update to subtract

    raise RuntimeError(f'{integrator.__name__}: Newton solve failed')

In [ ]:
# ------------------------------------------------------------------- TEST CELL (B2)
x1 = np.array([0.1, 0.2, 0.3, 0.4])
dt = 0.1
for method in implicit:
    x2 = np.asarray(implicit_integrator_solve(
        PARAMS, double_pendulum_dynamics, method, x1, dt), dtype=float).reshape(-1)
    norm = np.linalg.norm(np.asarray(
        method(PARAMS, double_pendulum_dynamics, ca.DM(x1), ca.DM(x2), dt)))
    assert norm < 1e-10, \
        f'{method.__name__}: residual is {norm:.3e} after the solve, Newton did not converge'
    assert np.allclose(x1, [0.1, 0.2, 0.3, 0.4]), \
        f'{method.__name__}: x1 was modified - you aliased x1 instead of copying it'
    print(f'{method.__name__:<20} x2 = {x2}   ||r|| = {norm:.2e}')

# A vanishingly short step must leave the state essentially where it started.
for method in implicit:
    x2 = np.asarray(implicit_integrator_solve(
        PARAMS, double_pendulum_dynamics, method, x1, 1e-9), dtype=float).reshape(-1)
    assert np.allclose(x2, x1, atol=1e-6), \
        f'{method.__name__}: a dt of 1e-9 should barely move the state'
print('\nPart B2 tests passed.')

Finally, the implicit counterpart of `simulate_explicit`. Each step now costs a Newton
solve instead of a function evaluation, which is the price implicit methods charge.

In [ ]:
def simulate_implicit(params, dynamics, integrator, x0, dt, tf, tol=1e-13,
                      energy_fn=double_pendulum_energy):
    """X, E = simulate_implicit(params, dynamics, integrator, x0, dt, tf)

    Roll the dynamics forward from `x0` to `tf`, solving the implicit `integrator`'s
    residual at every step.
    """
    times = time_grid(dt, tf)
    X = np.zeros((len(times), len(x0)))
    X[0] = 1 * np.asarray(x0, dtype=float)

    # given: build the symbolic residual + Jacobian graph ONCE, outside the loop
    functions = build_implicit_functions(params, dynamics, integrator, len(x0))

    # TODO: simulate X forward using implicit_integrator_solve
    # hint: pass tol=tol and functions=functions, so the graph is not rebuilt every step
    # DO NOT USE A WHILE LOOP

    E = np.array([float(energy_fn(params, x)) for x in X])      # given
    return X, E

In [ ]:
# ------------------------------------------------------------------- TEST CELL (B3)
# do not change dt, tf or the initial condition in this cell
dt, tf = 0.01, 2.0
for method in implicit:
    X, E = simulate_implicit(PARAMS, double_pendulum_dynamics, method, X0, dt, tf)
    results[method] = (X, E)
    assert X.shape == (201, 4), f'{method.__name__}: expected (201, 4), got {X.shape}'
    assert np.all(np.isfinite(X)), f'{method.__name__}: produced non-finite states'
    assert np.allclose(X[0], X0), f'{method.__name__}: X[0] is no longer x0'

dE = {method: E[-1] - E[0] for method, (_, E) in results.items()}
assert 2.5 < dE[forward_euler] < 3.0, 'forward euler should gain about 2.75 J'
assert -3.0 < dE[backward_euler] < -2.5, 'backward euler should lose about 2.73 J'
assert abs(dE[implicit_midpoint]) < 1e-2, f'implicit midpoint drifted {dE[implicit_midpoint]:.3e}'
assert abs(dE[hermite_simpson]) < 1e-4, f'hermite simpson drifted {dE[hermite_simpson]:.3e}'

print(f'{"integrator":<22}{"energy change (J)":>20}{"max energy error (J)":>24}')
for method, (_, E) in results.items():
    print(f'{method.__name__:<22}{E[-1] - E[0]:20.8g}{max_err_E(E):24.8g}')
print('\nPart B3 tests passed.')

fig, ax = plt.subplots(figsize=(9, 5))
for method, (_, E) in results.items():
    ax.plot(time_grid(dt, tf), E, '--' if method in implicit else '-',
            label=method.__name__.replace('_', ' ').title())
ax.set(xlabel='Time (s)', ylabel='Energy (J)', title='Energy behavior, dt = 0.01 s')
ax.legend()
fig.tight_layout()
plt.show()

Three qualitatively different behaviours, and only one of them is simply "small error":

- **Forward Euler gains energy.** The pendulum speeds up without bound. Run it long enough
  and the simulation blows up.
- **Backward Euler loses energy.** It looks stable and well behaved - and it is lying to
  you. It has quietly added damping to a frictionless system. A robot simulated this way
  will look easier to control than the real one.
- **Midpoint, implicit midpoint, RK4 and Hermite-Simpson** stay essentially flat. Implicit
  midpoint is *symplectic*: its energy error oscillates but does not drift, no matter how
  long you integrate. That property is why relatives of it show up inside physics engines,
  MuJoCo included.

So "stable" and "accurate" are not the same thing. Backward Euler's damping is precisely
what makes it popular for stiff systems - it refuses to explode - but you are paying for
that with physics you did not ask for.

### Part C (5 pts): Accuracy Comparison

So far we compared methods at one fixed time step. The more useful question is how the
error behaves as $\Delta t$ shrinks. A method is **order $p$** when its error scales like
$\mathcal{O}(\Delta t^p)$, so on a log-log plot of error against $\Delta t$ it is a straight
line of slope $p$. Halving $\Delta t$ cuts a 1st-order method's error in half, and a
4th-order method's by a factor of 16.

In [ ]:
# this cell is given, no TODOs here
dts = [1e-3, 1e-2, 1e-1]
fig, ax = plt.subplots(figsize=(9, 5))
print(f'{"integrator":<22}' + ''.join(f'{"dt=" + str(h):>15}' for h in dts))
for method in explicit + implicit:
    simulator = simulate_implicit if method in implicit else simulate_explicit
    errors = [max_err_E(simulator(PARAMS, double_pendulum_dynamics, method, X0, h, 2.0)[1])
              for h in dts]
    ax.loglog(dts, errors, 'o--' if method in implicit else 'o-',
              label=method.__name__.replace('_', ' ').title())
    print(f'{method.__name__:<22}' + ''.join(f'{e:15.4e}' for e in errors))
ax.set(xlabel='Time step (s)', ylabel='Maximum energy error (J)',
       title='Integrator comparison over 2 seconds')
ax.legend()
ax.grid(True, which='both', alpha=0.3)
fig.tight_layout()
plt.show()

Read the slopes off the plot: the two Euler methods fall roughly one decade per decade, the
two midpoint methods two, and RK4 and Hermite-Simpson four. Notice also that at any fixed
$\Delta t$ the implicit method beats the explicit method of the same order here, and that
the 4th-order methods at $\Delta t = 10^{-1}$ are already more accurate than Euler will ever
be at any step size you can afford.

#### Short answer (5 pts)

Answer in two or three sentences each.

**1.** Forward Euler systematically *gains* energy on this system while backward Euler
systematically *loses* it. Explain why, in terms of where each method's step lands relative
to the true trajectory.

**PUT YOUR ANSWER HERE**

**2.** RK4 costs four dynamics evaluations per step. Implicit midpoint costs three or four
Newton iterations, each needing a dynamics evaluation *and* a Jacobian factorisation - so it
is clearly more expensive per step. For a stiff, contact-rich model of the kind you will
simulate in MuJoCo, which would you choose, and what exactly are you buying?

**PUT YOUR ANSWER HERE**

**3.** From the log-log plot: give a time step at which RK4 is more accurate than implicit
midpoint, and describe a situation where you would still pick implicit midpoint over RK4
despite that.

**PUT YOUR ANSWER HERE**

### Part D (5 pts BONUS): Cart-pole

Here is the payoff for passing `params` and `dynamics` as arguments all the way through:
none of your integrators ever mentions the double pendulum, so switching to a completely
different robot should cost you nothing but different arguments.

The cart-pole is the system we will keep coming back to in this course. Its state is

$$x = \begin{bmatrix} p & \dot{p} & \theta & \dot\theta \end{bmatrix}^T$$

with $p$ the cart position and $\theta = 0$ pointing straight up. We start it just off
vertical with no force applied, so it falls over.

In [ ]:
# these functions are given, no TODOs here

@dataclass(frozen=True)
class CartPoleParameters:
    M: float = 1.0    # cart mass (kg)
    m: float = 0.1    # point mass at the pole tip (kg)
    L: float = 1.0    # massless pole length (m)
    g: float = 9.8


CARTPOLE_PARAMS = CartPoleParameters()
CARTPOLE_X0 = np.array([0.0, 0.0, 0.1, 0.0])


def cartpole_dynamics(params, x, force=0.0):
    """State [position, velocity, theta, omega]; theta = 0 is upright.

    Positive theta tilts the pole toward positive cart position. Force acts horizontally
    on the cart (N). This experiment uses zero force, no friction and no controller.
    """
    velocity, theta, omega = x[1], x[2], x[3]
    M, m, L, g = params.M, params.m, params.L, params.g
    s, c = ca.sin(theta), ca.cos(theta)
    acceleration = (force + m * s * (L * omega**2 - g * c)) / (M + m * s**2)
    alpha = (g * s - c * acceleration) / L
    return ca.vertcat(velocity, acceleration, omega, alpha)


def cartpole_energy(params, x):
    """Mechanical energy with potential zero at cart height."""
    velocity, theta, omega = x[1], x[2], x[3]
    M, m, L, g = params.M, params.m, params.L, params.g
    return (0.5 * (M + m) * velocity**2 + 0.5 * m * L**2 * omega**2
            + m * L * velocity * omega * ca.cos(theta) + m * g * L * ca.cos(theta))


# These already pass - they are here to tell you the model is trustworthy before you
# start blaming it for your integrator's behaviour.
np.testing.assert_allclose(cartpole_dynamics(CARTPOLE_PARAMS, np.zeros(4)), 0, atol=1e-14)
assert float(cartpole_dynamics(CARTPOLE_PARAMS, CARTPOLE_X0)[3]) > 0   # tips away from upright

x, u = ca.SX.sym('x', 4), ca.SX.sym('u')
energy_rate = ca.Function('energy_rate', [x, u],
                          [ca.jacobian(cartpole_energy(CARTPOLE_PARAMS, x), x)
                           @ cartpole_dynamics(CARTPOLE_PARAMS, x, u)])
state, force = np.array([0.2, 0.3, 0.4, -0.5]), 1.7
# power balance: energy only enters through the force doing work on the cart
np.testing.assert_allclose(float(energy_rate(state, force)), force * state[1], atol=1e-12)
print('cart-pole model checks passed.')

In [ ]:
cartpole_results = {}

# TODO: fill cartpole_results[method] = (X, E) for all six integrators, using
# dt = 0.01 and tf = 2.0 from CARTPOLE_X0.
# Only three things change from Part B: params, dynamics, and energy_fn.
# hint: `simulate_implicit if method in implicit else simulate_explicit` picks the simulator
# DO NOT USE A WHILE LOOP

In [ ]:
# ------------------------------------------------------------------- TEST CELL (D)
# If this cell fails, the usual cause is that simulate_explicit or simulate_implicit
# ignores its `dynamics` or `energy_fn` argument and hardcodes the double pendulum.
state = np.array([0.2, 0.3, 0.4, -0.5])
for method in implicit:
    x2 = np.asarray(implicit_integrator_solve(
        CARTPOLE_PARAMS, cartpole_dynamics, method, state, 0.1), dtype=float).reshape(-1)
    norm = np.linalg.norm(np.asarray(
        method(CARTPOLE_PARAMS, cartpole_dynamics, ca.DM(state), ca.DM(x2), 0.1)))
    assert norm < 1e-10, f'{method.__name__}: residual {norm:.3e} on the cart-pole'

assert set(cartpole_results) == set(explicit + implicit), 'all six integrators are required'
for method, (X, E) in cartpole_results.items():
    assert X.shape == (201, 4), f'{method.__name__}: expected (201, 4), got {X.shape}'
    assert np.all(np.isfinite(X)) and np.all(np.isfinite(E)), f'{method.__name__}: non-finite'
    np.testing.assert_array_equal(X[0], CARTPOLE_X0)

dE = {method: E[-1] - E[0] for method, (_, E) in cartpole_results.items()}
assert max_err_E(cartpole_results[rk4][1]) < 1e-5, 'rk4 should conserve energy to ~1e-8 here'
assert abs(dE[forward_euler]) > 1e-2, 'forward euler should still visibly inject energy'

print(f'{"integrator":<22}{"energy change (J)":>20}{"max energy error (J)":>24}')
for method, (_, E) in cartpole_results.items():
    print(f'{method.__name__:<22}{E[-1] - E[0]:20.8g}{max_err_E(E):24.8g}')
print('\nPart D tests passed.')

fig, ax = plt.subplots(figsize=(9, 5))
for method, (_, E) in cartpole_results.items():
    ax.plot(time_grid(0.01, 2.0), E, '--' if method in implicit else '-',
            label=method.__name__.replace('_', ' ').title())
ax.set(xlabel='Time (s)', ylabel='Energy (J)', title='Cart-pole energy behavior, dt = 0.01 s')
ax.legend()
fig.tight_layout()
plt.show()

In [ ]:
display(show_animation(animate_cartpole(CARTPOLE_PARAMS, cartpole_results[rk4][0], 0.01,
                                        title='RK4: unforced cart-pole')))

### Submitting

1. **Kernel -> Restart Kernel and Run All Cells.** A notebook that only passes when run out
   of order does not pass for us.
2. Check that every test cell printed its `... tests passed.` line.
3. Check that you answered all three short-answer questions in Part C.
4. Export to PDF: `File -> Save and Export Notebook As -> HTML`, then print that HTML to PDF
   from your browser. The animations will not appear in the PDF; that is expected.
5. Submit the PDF **and** the `.ipynb`.